Link Dataset: https://www.kaggle.com/datasets/tongpython/cat-and-dog/data?select=test_set

In [ ]:
# Import Libraries That We Need For
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn as nn
import kagglehub
import torch
import os

In [ ]:
# Get The Path Of Dataset
path = kagglehub.dataset_download('tongpython/cat-and-dog')

Using Colab cache for faster access to the 'cat-and-dog' dataset.


In [ ]:
# Checking The Device Use Cuda If Available If Not Available Use CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Path Folder The Image Of 'training_set' and 'test_set'
train_dir = os.path.join(path, 'training_set', 'training_set')
test_dir = os.path.join(path, 'test_set', 'test_set')

In [ ]:
# Transformation Image
train_transform = transforms.Compose([
    # Data Augmentation
    transforms.RandomRotation(degrees=20),
    transforms.RandomResizedCrop(size=(128, 128), scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Loading data with ImageFolder
train_dataset = datasets.ImageFolder(train_dir, transform = train_transform)
test_dataset = datasets.ImageFolder(test_dir, transform = test_transform)

In [ ]:
# DataLoader for batching processing
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
print(f"Jumlah gambar training: {len(train_dataset)}")
print(f"Kelas yang terdeteksi: {train_dataset.classes}")

Jumlah gambar training: 8005
Kelas yang terdeteksi: ['cats', 'dogs']


In [ ]:
from torch.nn.modules.linear import Linear
from torch.nn.modules.dropout import Dropout
# Making The Model CNN
class CNNClassifier(nn.Module):
    def __init__(self, num_classes=2) -> None:
        super(CNNClassifier, self).__init__()

        self.features = nn.Sequential(
            # Block 1: Input (3, 128, 128) -> Output (32, 64, 64)
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 2: Input (32, 64, 64) -> Output (64, 32, 32)
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 3: Input (64, 32, 32) -> Output (128, 16, 16)
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
model = CNNClassifier(num_classes=len(train_dataset.classes)).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        # Forward Pass
        outputs = model.forward(images)
        loss = criterion(outputs, labels)

        # Backward Pass & Optimizer Step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Static
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [ ]:
epochs = 5
for epoch in range(epochs):
    loss, acc = train_epoch(model, train_loader, criterion, optimizer)
    print(f"{epoch+1}/epochs | Loss:{loss:.4f} | Accuracy {acc*100:.2f}%")

1/epochs | Loss:0.6697 | Accuracy 59.14%
2/epochs | Loss:0.6020 | Accuracy 66.97%
3/epochs | Loss:0.5577 | Accuracy 71.61%
4/epochs | Loss:0.5282 | Accuracy 73.98%
5/epochs | Loss:0.4993 | Accuracy 76.15%
